<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/03_celestial_visibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Celestial Visibility Engine

Computes which deep-sky objects (NGC/IC catalog, 13,000+ objects) and
solar system bodies (planets + Moon) are realistically observable from
the user's location across the upcoming 7-night window.

Unlike the weather module, visibility is pure orbital mechanics — no
forecast horizon, all 7 nights get full data.

### Pipeline
1. **Type filter** — keeps only meaningful deep-sky object types (galaxies,
   clusters, nebulae etc.), removes stars, double stars, asterisms, errors
2. **Magnitude filter** — drops objects too faint for the user's telescope aperture
3. **Declination filter** — drops objects that never reach >30° altitude from this latitude
4. **Nightly sky computation** — vectorized observability check across all
   candidates at once, then detailed altitude/timing computed only for
   objects that pass, plus planet positions computed separately
5. **Composite ranking** — objects ranked by combined altitude + brightness
   score with per-type diversity cap (max 100 objects per night)

### Inputs (from Drive)
- `current_request.json` — user profile (location, telescope specs)

### Outputs (to Drive)
- `weekly_visibility.json` — visible objects + planets per night for
   all 7 days, each with peak altitude, rise/set times, moon separation,
   magnitude, object type, and composite score

### Notes
- Observer and target list built once and reused across all 7 nights
- Moon position computed at each object's personal peak time (not midnight)
  to avoid ~3° drift error for objects near the 30° separation threshold
- Night window gracefully returns empty if no astronomical darkness exists
  (relevant for high latitudes in summer)
- Planets computed separately via Astropy ephemeris, not from NGC catalog

## Setup

In [37]:
!pip install astropy astroplan requests -q

import sys, os, json, requests, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
from typing import Optional
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git 2>/dev/null || git -C /content/AstroPlanner-AI pull

sys.path.append('/content/AstroPlanner-AI/src')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'
os.makedirs(DATA_DIR, exist_ok=True)

##Imports

In [38]:
from astropy.coordinates import (
    EarthLocation, SkyCoord, AltAz, get_body
)
from astropy.time import Time
import astropy.units as u
from astropy.utils.iers import conf as iers_conf
from astroplan import Observer, FixedTarget
from astroplan import (
    AltitudeConstraint,
    MoonSeparationConstraint,
    is_observable,
)

from models import WeeklyPlanRequest, UserProfile

## Suppress warnings

In [39]:
# Disable IERS auto-download — Colab blocks it, bundled table is
# accurate enough for observing planning (arcsecond-level precision).
iers_conf.auto_download = False
iers_conf.auto_max_age = None

# Suppress the NonRotationTransformation warning that fires inside
# MoonSeparationConstraint's internals — results are still correct.
warnings.filterwarnings('ignore', message='.*IERS.*')
warnings.filterwarnings('ignore', message='.*NonRotation.*')
warnings.filterwarnings('ignore', message='.*failed to download.*')
warnings.filterwarnings('ignore', message='.*Angular separation.*')
warnings.filterwarnings('ignore', message='.*unable to download.*')

## Load request

In [40]:
with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

print(f"Computing visibility for: {plan_request.user.name}")
print(f"Location: {plan_request.user.latitude}°N, {plan_request.user.longitude}°E")
print(f"Telescope aperture: {plan_request.user.telescope.aperture_mm}mm")

Computing visibility for: Arfa
Location: 33.223°N, 32.44°E
Telescope aperture: 64.0mm


## Load and prefilter NGC catalog

In [41]:
# Deep-sky object types worth recommending to observers.
# Excludes: individual stars (*), double stars (Dbl*, D*), asterisms (Ast),
# multiple stars (Mult*, ***), non-existing objects (NonEx), errors (Err).
USEFUL_OBJECT_TYPES = {
    'GX',    # Galaxy
    'OC',    # Open cluster
    'GC',    # Globular cluster
    'BN',    # Bright nebula (generic)
    'EN',    # Emission nebula
    'RN',    # Reflection nebula
    'PN',    # Planetary nebula
    'SNR',   # Supernova remnant
    'SC',    # Star cloud
    'CL+N',  # Cluster with nebula
    'G+C',   # Galaxy + cluster
    'dup',   # Duplicate entry (keep — catalog deduplication handles it)
}

def load_ngc_catalog() -> pd.DataFrame:
    """
    Downloads and loads the OpenNGC catalog from GitHub — a clean,
    maintained CSV of all NGC/IC objects with coordinates, magnitudes,
    object types, and sizes. Cached to Drive after first download.
    """
    cache_path = f'{DATA_DIR}/ngc_catalog.csv'

    if not os.path.exists(cache_path):
        print("Downloading OpenNGC catalog (first time only)...")
        url = "https://raw.githubusercontent.com/mattiaverga/OpenNGC/master/database_files/NGC.csv"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(cache_path, 'w') as f:
            f.write(response.text)
        print(f"Saved to {cache_path}")
    else:
        print("Loaded NGC catalog from cache.")

    df = pd.read_csv(cache_path, sep=';', low_memory=False)

    cols = ['Name', 'Type', 'RA', 'Dec', 'V-Mag', 'B-Mag', 'MajAx', 'Common names']
    df = df[[c for c in cols if c in df.columns]]

    # Prefer V-Mag (visual), fall back to B-Mag (blue) if missing
    df['magnitude'] = pd.to_numeric(df['V-Mag'], errors='coerce')
    df.loc[df['magnitude'].isna(), 'magnitude'] = pd.to_numeric(
        df.loc[df['magnitude'].isna(), 'B-Mag'], errors='coerce'
    )

    print(f"Total objects in catalog: {len(df)}")
    return df


def compute_limiting_magnitude(aperture_mm: float) -> float:
    """
    Estimates the visual limiting magnitude achievable with this telescope
    aperture using the standard formula: 7.5 + 5 * log10(aperture_mm).
    Conservative visual estimate; imaging can go ~2 magnitudes deeper but
    the recommendation engine handles that distinction downstream.
    """
    return 7.5 + 5 * np.log10(aperture_mm)


def prefilter_catalog(df: pd.DataFrame, user: UserProfile) -> pd.DataFrame:
    """
    Three-stage cheap filtering before any expensive sky computation:

    Stage 1 — Type filter: keeps only meaningful deep-sky object types,
    removing individual stars, double stars, asterisms, non-existing
    objects and catalog errors from the NGC catalog.

    Stage 2 — Magnitude filter: drops objects fainter than the telescope's
    limiting magnitude. Objects with no magnitude data are kept (some
    bright nebulae have unreliable catalog magnitudes).

    Stage 3 — Declination filter: drops objects that can never reach >=30°
    altitude from this latitude. At latitude φ, an object at declination δ
    has maximum possible altitude = 90° - |φ - δ|.
    """
    lat = user.latitude
    limiting_mag = compute_limiting_magnitude(user.telescope.aperture_mm)
    print(f"Telescope limiting magnitude: {limiting_mag:.1f}")

    def parse_ra(ra_str):
        try:
            h, m, s = str(ra_str).split(':')
            return float(h) * 15 + float(m) * 0.25 + float(s) * (15 / 3600)
        except Exception:
            return np.nan

    def parse_dec(dec_str):
        try:
            parts = str(dec_str).split(':')
            sign = -1 if str(dec_str).startswith('-') else 1
            return sign * (abs(float(parts[0])) + float(parts[1]) / 60 + float(parts[2]) / 3600)
        except Exception:
            return np.nan

    df = df.copy()
    df['ra_deg'] = df['RA'].apply(parse_ra)
    df['dec_deg'] = df['Dec'].apply(parse_dec)
    df = df.dropna(subset=['ra_deg', 'dec_deg'])

    total = len(df)

    # Stage 1: object type filter (NEW — was missing before)
    df = df[df['Type'].isin(USEFUL_OBJECT_TYPES)]
    print(f"After type filter:      {len(df):5d} objects (removed {total - len(df)})")

    # Stage 2: magnitude filter
    before = len(df)
    df = df[df['magnitude'].isna() | (df['magnitude'] <= limiting_mag)]
    print(f"After magnitude filter: {len(df):5d} objects (removed {before - len(df)})")

    # Stage 3: declination / max altitude filter
    before = len(df)
    df['max_altitude'] = 90 - abs(lat - df['dec_deg'])
    df = df[df['max_altitude'] >= 30]
    print(f"After declination filter:{len(df):5d} objects (removed {before - len(df)})")

    return df.reset_index(drop=True)


ngc_df = load_ngc_catalog()
filtered_df = prefilter_catalog(ngc_df, plan_request.user)
print(f"\nFinal candidate count for nightly sky computation: {len(filtered_df)}")

Loaded NGC catalog from cache.
Total objects in catalog: 13969
Telescope limiting magnitude: 16.5
After type filter:        141 objects (removed 13821)
After magnitude filter:   141 objects (removed 0)
After declination filter:   95 objects (removed 46)

Final candidate count for nightly sky computation: 95


## Build observer and targets once

In [42]:
def build_observer(user: UserProfile) -> Observer:
    """
    Builds the Astroplan Observer for this user's location. Built once
    and reused across all 7 nights — location doesn't change between nights,
    so rebuilding it 7 times was pure waste.
    """
    return Observer(
        location=EarthLocation(
            lat=user.latitude * u.deg,
            lon=user.longitude * u.deg,
            height=0 * u.m
        ),
        name=f"observer_{user.name}"
    )


def build_target_list(filtered_df: pd.DataFrame) -> tuple[list, list]:
    """
    Converts the filtered catalog DataFrame into Astroplan FixedTarget objects.
    Built once and reused across all 7 nights — the catalog doesn't change
    between nights. Returns (targets, valid_rows) as parallel lists.
    """
    targets = []
    valid_rows = []
    for _, row in filtered_df.iterrows():
        try:
            coord = SkyCoord(ra=row['ra_deg'] * u.deg, dec=row['dec_deg'] * u.deg)
            targets.append(FixedTarget(coord=coord, name=str(row['Name'])))
            valid_rows.append(row)
        except Exception:
            continue
    print(f"Built {len(targets)} FixedTarget objects (built once, reused across all 7 nights)")
    return targets, valid_rows


observer = build_observer(plan_request.user)
targets, valid_rows = build_target_list(filtered_df)

Built 95 FixedTarget objects (built once, reused across all 7 nights)


## Night window helper

In [43]:
def get_night_window(observer: Observer, date_str: str):
    """
    Returns (night_start, night_end) as Astropy Time objects for
    astronomical twilight on the given date. Returns (None, None) if no
    astronomical darkness exists for this location/date — relevant for
    high latitudes (>~48°N) in summer where the sun never drops to -18°.
    """
    try:
        midnight = Time(f"{date_str} 23:59:00")
        night_start = observer.twilight_evening_astronomical(midnight, which='nearest')
        night_end = observer.twilight_morning_astronomical(midnight, which='nearest')
        duration = (night_end - night_start).to(u.hour).value
        if duration <= 0:
            return None, None
        return night_start, night_end
    except Exception:
        return None, None  # no astronomical darkness this night

## Planet visibility

In [44]:
# Solar system bodies to check alongside the NGC catalog.
# Excludes Sun (dangerous to observe without proper filters, edge case).
PLANETS = [
    ('moon',    'Moon',    'Solar System', None),
    ('jupiter', 'Jupiter', 'Planet',       -2.9),
    ('saturn',  'Saturn',  'Planet',        0.7),
    ('mars',    'Mars',    'Planet',        1.0),
    ('venus',   'Venus',   'Planet',       -4.5),
    ('mercury', 'Mercury', 'Planet',       -0.5),
]


def get_planet_visibility(
    observer: Observer,
    user: UserProfile,
    date_str: str,
    night_start,
    night_end,
    n_steps: int,
    time_grid,
    time_labels: list[str],
) -> list[dict]:
    """
    Computes visibility of solar system bodies (planets + Moon) for the
    given night. Uses the same time grid as the NGC computation for
    consistency. Moon is always included when above 30° — it's relevant
    context for planning even if it washes out faint objects nearby.
    Planets are filtered to >= 30° altitude during the night window,
    same as NGC objects, so the recommendation engine can treat them
    uniformly.
    """
    midnight = Time(f"{date_str} 23:59:00")
    planet_results = []

    for body_name, display_name, obj_type, typical_mag in PLANETS:
        try:
            # Compute position across the night grid
            altaz_frame = AltAz(obstime=time_grid, location=observer.location)
            body = get_body(body_name, time_grid, ephemeris='builtin')
            body_altaz = body.transform_to(altaz_frame)
            alts = body_altaz.alt.deg

            peak_idx = int(np.argmax(alts))
            peak_alt = float(alts[peak_idx])

            # Skip if never above 30° during the night
            if peak_alt < 30:
                continue

            above_threshold = alts >= 30
            rising_indices = np.where(above_threshold)[0]
            rise_time = time_labels[rising_indices[0]] if len(rising_indices) > 0 else "circumpolar"
            set_time  = time_labels[rising_indices[-1]] if len(rising_indices) > 0 else "circumpolar"

            # Moon separation from this planet at its peak time
            moon_body = get_body('moon', time_grid[peak_idx], ephemeris='builtin')
            moon_coord = SkyCoord(
                ra=moon_body.ra.deg * u.deg,
                dec=moon_body.dec.deg * u.deg,
                frame='icrs'
            )
            planet_coord = SkyCoord(
                ra=float(body_altaz.data.lon.deg[peak_idx]) * u.deg if body_name != 'moon' else 0 * u.deg,
                dec=float(body_altaz.data.lat.deg[peak_idx]) * u.deg if body_name != 'moon' else 0 * u.deg,
                frame='icrs'
            )

            # For moon, just use its own ra/dec at peak
            body_at_peak = get_body(body_name, time_grid[peak_idx], ephemeris='builtin')
            body_icrs = SkyCoord(
                ra=body_at_peak.ra.deg * u.deg,
                dec=body_at_peak.dec.deg * u.deg,
                frame='icrs'
            )
            moon_sep = float(moon_coord.separation(body_icrs).deg) if body_name != 'moon' else None

            planet_results.append({
                "name": display_name,
                "common_name": display_name,
                "type": obj_type,
                "magnitude": typical_mag,
                "size_arcmin": None,  # planets' apparent size varies; not relevant for ranking
                "ra_deg": round(float(body_at_peak.ra.deg), 4),
                "dec_deg": round(float(body_at_peak.dec.deg), 4),
                "peak_altitude_deg": round(peak_alt, 1),
                "peak_time_local": time_labels[peak_idx],
                "rise_time_local": rise_time,
                "set_time_local": set_time,
                "moon_separation_deg": round(moon_sep, 1) if moon_sep is not None else None,
                "is_solar_system": True,
            })

        except Exception:
            continue

    return planet_results

## Nightly visibility computation

In [45]:
def compute_nightly_visibility(
    observer: Observer,
    targets: list,
    valid_rows: list,
    date_str: str,
    user: UserProfile,
) -> list[dict]:
    """
    Computes visible objects for one night. Key improvements over v1:
    - Observer and targets passed in (built once externally, not rebuilt here)
    - Moon position computed at each object's own peak time, not midnight
      (fixes ~3° drift error for objects near the 30° separation threshold)
    - Composite ranking: 50% altitude + 50% brightness (not altitude-only)
    - Per-type diversity cap so results span galaxies, clusters, nebulae
    - Night window gracefully returns [] if no astronomical darkness
    - Planets computed separately and merged into results
    """
    night_start, night_end = get_night_window(observer, date_str)
    if night_start is None:
        return []

    duration_hours = (night_end - night_start).to(u.hour).value
    n_steps = max(int(duration_hours * 2), 2)
    time_grid = night_start + np.linspace(0, duration_hours, n_steps) * u.hour
    time_labels = [t.to_datetime().strftime('%H:%M') for t in time_grid]

    constraints = [
        AltitudeConstraint(min=30 * u.deg),
        MoonSeparationConstraint(min=30 * u.deg),
    ]

    # Single vectorized observability check across all targets
    observable_mask = is_observable(
        constraints, observer, targets,
        time_range=[night_start, night_end]
    )

    visible = []
    for target, row, is_obs in zip(targets, valid_rows, observable_mask):
        if not is_obs:
            continue
        try:
            coord = target.coord
            altaz_frame = AltAz(obstime=time_grid, location=observer.location)
            altaz = coord.transform_to(altaz_frame)
            alts = altaz.alt.deg

            peak_idx = int(np.argmax(alts))
            peak_alt = float(alts[peak_idx])
            peak_time = time_labels[peak_idx]

            # Rise/set derived from altitude grid — no extra Astroplan calls
            above_threshold = alts >= 30
            rising_indices = np.where(above_threshold)[0]
            rise_time = time_labels[rising_indices[0]]  if len(rising_indices) > 0 else "circumpolar"
            set_time  = time_labels[rising_indices[-1]] if len(rising_indices) > 0 else "circumpolar"

            # Moon separation at THIS object's peak time (not midnight)
            # Fixes ~3° positional drift error for objects near the threshold
            moon_at_peak = get_body('moon', time_grid[peak_idx], ephemeris='builtin')
            moon_coord = SkyCoord(
                ra=moon_at_peak.ra.deg * u.deg,
                dec=moon_at_peak.dec.deg * u.deg,
                frame='icrs'
            )
            moon_sep = float(moon_coord.separation(coord).deg)

            mag = row['magnitude']
            common = str(row.get('Common names', '') or '').split(';')[0].strip() or None

            visible.append({
                "name": str(row['Name']),
                "common_name": common,
                "type": str(row.get('Type', 'Unknown')),
                "magnitude": float(mag) if not pd.isna(mag) else None,
                "size_arcmin": float(row['MajAx']) if 'MajAx' in row and not pd.isna(row.get('MajAx')) else None,
                "ra_deg": round(float(row['ra_deg']), 4),
                "dec_deg": round(float(row['dec_deg']), 4),
                "peak_altitude_deg": round(peak_alt, 1),
                "peak_time_local": peak_time,
                "rise_time_local": rise_time,
                "set_time_local": set_time,
                "moon_separation_deg": round(moon_sep, 1),
                "is_solar_system": False,
            })

        except Exception:
            continue

    # Composite score: 50% altitude + 50% brightness
    # Prevents faint-but-high objects from dominating the ranking
    for obj in visible:
        brightness_score = max(0, (15 - (obj['magnitude'] or 15)) / 15)
        altitude_score = obj['peak_altitude_deg'] / 90
        obj['_sort_score'] = 0.5 * altitude_score + 0.5 * brightness_score

    visible.sort(key=lambda x: x['_sort_score'], reverse=True)

    # Per-type diversity cap — ensures recommendation engine gets a
    # spread of object types, not 100 galaxies and nothing else
    TYPE_CAPS = {
        'GX': 30, 'OC': 15, 'GC': 15,
        'BN': 8,  'EN': 8,  'RN': 5,
        'PN': 10, 'SNR': 5, 'SC': 3,
        'CL+N': 5, 'G+C': 3, 'dup': 3,
    }
    type_counts = defaultdict(int)
    diverse_visible = []
    for obj in visible:
        obj_type = obj['type']
        cap = TYPE_CAPS.get(obj_type, 5)
        if type_counts[obj_type] < cap:
            diverse_visible.append(obj)
            type_counts[obj_type] += 1
        if len(diverse_visible) >= 100:
            break

    # Clean up internal sort score before saving
    for obj in diverse_visible:
        obj.pop('_sort_score', None)

    # Add planets — computed separately, always prepended since they
    # are typically the highest-priority targets (bright, easy to find)
    planets = get_planet_visibility(
        observer, user, date_str,
        night_start, night_end,
        n_steps, time_grid, time_labels
    )

    return planets + diverse_visible

## Weekly loop

In [46]:
def get_weekly_visibility(
    user: UserProfile,
    observer: Observer,
    targets: list,
    valid_rows: list,
) -> list[dict]:
    """
    Computes visible objects for all 7 nights. Observer and target list
    are passed in (built once in Cell 6) and reused across every night —
    the single biggest performance improvement over v1.
    """
    today = datetime.today()
    weekly_visibility = []

    for offset in range(7):
        date_str = (today + timedelta(days=offset)).strftime('%Y-%m-%d')
        print(f"Computing visibility for {date_str}...", end=" ")
        nightly = compute_nightly_visibility(observer, targets, valid_rows, date_str, user)
        print(f"{len(nightly)} objects ({sum(1 for o in nightly if o.get('is_solar_system'))} planets/moon + {sum(1 for o in nightly if not o.get('is_solar_system'))} deep-sky)")

        weekly_visibility.append({
            "date": date_str,
            "day_offset": offset,
            "visible_object_count": len(nightly),
            "objects": nightly,
        })

    return weekly_visibility

## Run and save

In [47]:
weekly_visibility = get_weekly_visibility(plan_request.user, observer, targets, valid_rows)

with open(f'{DATA_DIR}/weekly_visibility.json', 'w') as f:
    json.dump(weekly_visibility, f, indent=2, default=str)

print(f"\nSaved to {DATA_DIR}/weekly_visibility.json")
print("\n=== Visibility Summary ===")
for night in weekly_visibility:
    print(f"{night['date']}: {night['visible_object_count']} objects visible")

Computing visibility for 2026-07-01... 

 2461223.36506947 2461223.3859028  2461223.40673613 2461223.42756947
 2461223.4484028  2461223.46923613 2461223.49006947 2461223.5109028
 2461223.53173613], obsgeoloc=[(-3178458.1543189 , -4285449.41746731, 3483043.24592428),
 (-2590142.37193748, -4665766.21894962, 3481530.02063872),
 (-1957111.00825898, -4965811.64107173, 3479898.62613181),
 (-1290254.70939359, -5180423.71395648, 3478177.12748661),
 ( -601046.04430471, -5305910.25969955, 3476395.14009078),
 (   98657.86881691, -5340112.41245297, 3474583.32015398),
 (  796819.35194685, -5282441.75946774, 3472772.83731608),
 ( 1481427.26296496, -5133890.46412348, 3470994.83841912),
 ( 2140703.6348719 , -4897014.19678862, 3469279.91166798),
 ( 2763306.30346935, -4575888.16716764, 3467657.56039834),
 ( 3338524.03750465, -4176037.01455438, 3466155.69550494),
 ( 3856460.81377084, -3704339.76267726, 3464800.15526208),
 ( 4308206.06915027, -3168911.47234139, 3463614.26079838)] m, obsgeovel=[(312.50616856, -232.43408324, -0.80295498),
 (340.23

17 objects (2 planets/moon + 15 deep-sky)
Computing visibility for 2026-07-02... 

 2461224.36493377 2461224.38576711 2461224.40660044 2461224.42743377
 2461224.44826711 2461224.46910044 2461224.48993377 2461224.51076711
 2461224.53160044], obsgeoloc=[(-3107977.22688074, -4336980.10968595, 3482864.37961417),
 (-2513523.21912655, -4707629.13515326, 3481334.82362354),
 (-1875671.75973795, -4997286.55976356, 3479690.45904001),
 (-1205396.41961702, -5200969.12832242, 3477959.57412066),
 ( -514228.58883071, -5315172.69366573, 3476171.94573937),
 (  185940.90896901, -5337932.50186891, 3474358.32712594),
 (  883066.38580572, -5268856.99379564, 3472549.9188056 ),
 ( 1565154.52291858, -5109134.54146576, 3470777.83184043),
 ( 2220470.70334595, -4861513.00334963, 3469072.55260694),
 ( 2837740.89381139, -4530252.45031953, 3467463.41831755),
 ( 3406345.60274741, -4121051.87555847, 3465978.11230905),
 ( 3916502.57707814, -3640951.14987702, 3464642.18778151),
 ( 4359435.09703564, -3098209.90707433, 3463478.62817809)] m, obsgeovel=[(316.26389161, -227.29461595, -0.81291009),
 (343.2

17 objects (2 planets/moon + 15 deep-sky)
Computing visibility for 2026-07-03... 

...

17 objects (2 planets/moon + 15 deep-sky)
Computing visibility for 2026-07-06... 

 2461228.36412967 2461228.38496301 2461228.40579634 2461228.42662968
 2461228.44746301 2461228.46829634 2461228.48912968 2461228.50996301
 2461228.53079634], obsgeoloc=[(-2825375.65964159, -4526720.23475995, 3482142.94645204),
 (-2208518.90404354, -4858750.05884795, 3480553.8808548 ),
 (-1553511.92987221, -5107188.36691102, 3478863.44515554),
 ( -871623.45096452, -5267761.03618838, 3477100.72139854),
 ( -174584.64949647, -5337705.58088588, 3475296.03535382),
 (  525612.64680928, -5315818.67787187, 3473480.43479573),
 ( 1216922.27161195, -5202476.86860064, 3471685.15535844),
 ( 1887450.9617214 , -4999630.0811077 , 3469941.08315775),
 ( 2525662.96832442, -4710768.0835254 , 3468278.22342429),
 ( 3120578.51755072, -4340860.44625007, 3466725.18428972),
 ( 3661962.70606634, -3896271.04565028, 3465308.68460698),
 ( 4140501.58206362, -3384648.58004599, 3464053.09427083),
 ( 4547962.38183215, -2814794.98203433, 3462980.01495244)] m, obsgeovel=[(330.10007396, -206.68712789, -0.84950271),
 (354.3

17 objects (2 planets/moon + 15 deep-sky)
Computing visibility for 2026-07-07... 

 2461229.36386386 2461229.38469719 2461229.40553053 2461229.42636386
 2461229.44719719 2461229.46803053 2461229.48886386 2461229.50969719
 2461229.53053053], obsgeoloc=[(-2754746.64387937, -4570183.81552278, 3481961.23208961),
 (-2132808.90599373, -4892595.77995056, 3480358.58869885),
 (-1474023.4527263 , -5130833.95241552, 3478657.93301942),
 ( -789724.00268257, -5280799.69302418, 3476888.52325593),
 (  -91683.21625302, -5339912.99730309, 3475080.80057097),
 (  608089.84076828, -5307156.88249321, 3473265.86537302),
 ( 1297556.30067672, -5183094.88368076, 3471474.9422605 ),
 ( 1964854.61005801, -4969861.35876196, 3469738.8428273 ),
 ( 2598504.59548423, -4671124.76903406, 3468087.43557164),
 ( 3187604.96796121, -4292024.56713268, 3466549.13202807),
 ( 3722020.86827264, -3839082.77809197, 3465150.39796351),
 ( 4192558.22705019, -3320091.79423946, 3463915.29804404),
 ( 4591121.93833448, -2743980.31586982, 3462865.08181721)] m, obsgeovel=[(333.26945819, -201.53679751, -0.85793281),
 (356.7

17 objects (2 planets/moon + 15 deep-sky)

Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_visibility.json

=== Visibility Summary ===
2026-07-01: 17 objects visible
2026-07-02: 17 objects visible
2026-07-03: 17 objects visible
2026-07-04: 17 objects visible
2026-07-05: 17 objects visible
2026-07-06: 17 objects visible
2026-07-07: 17 objects visible


 ## Spot check

In [48]:
best_night = max(weekly_visibility, key=lambda n: n['visible_object_count'])
print(f"Best night: {best_night['date']} ({best_night['visible_object_count']} objects)\n")

print("=== Solar System Bodies ===")
for obj in best_night['objects']:
    if obj.get('is_solar_system'):
        print(
            f"  {obj['name']:10} alt={obj['peak_altitude_deg']}° "
            f"at {obj['peak_time_local']}  "
            f"rise={obj['rise_time_local']} set={obj['set_time_local']}"
        )

print("\n=== Top 15 Deep-Sky Objects (by composite score) ===")
dso = [o for o in best_night['objects'] if not o.get('is_solar_system')]
for obj in dso[:15]:
    print(
        f"  {obj['name']:10} {(obj['common_name'] or ''):25} "
        f"type={obj['type']:5} mag={str(obj['magnitude']):5} "
        f"peak={obj['peak_altitude_deg']}° at {obj['peak_time_local']} "
        f"moon_sep={obj['moon_separation_deg']}°"
    )

print("\n=== Object Type Distribution (best night) ===")
type_dist = defaultdict(int)
for obj in best_night['objects']:
    type_dist[obj['type']] += 1
for t, count in sorted(type_dist.items(), key=lambda x: -x[1]):
    print(f"  {t:8}: {count}")

Best night: 2026-07-01 (17 objects)

=== Solar System Bodies ===
  Moon       alt=33.8° at 23:19  rise=22:11 set=01:02
  Saturn     alt=37.9° at 01:02  rise=00:28 set=01:02

=== Top 15 Deep-Sky Objects (by composite score) ===
  NGC6960    Veil Nebula,Filamentary Nebula,Western Veil type=SNR   mag=7.0   peak=87.4° at 23:54 moon_sep=53.1°
  NGC6992    Eastern Veil,Network Nebula type=SNR   mag=7.0   peak=86.9° at 23:54 moon_sep=54.6°
  NGC6995    Eastern Veil,Network Nebula type=SNR   mag=7.0   peak=86.5° at 23:54 moon_sep=54.2°
  NGC6720    Ring Nebula               type=PN    mag=8.8   peak=88.6° at 22:11 moon_sep=58.7°
  NGC6853    Dumbbell Nebula           type=PN    mag=7.4   peak=79.4° at 23:19 moon_sep=45.2°
  NGC7027    nan                       type=PN    mag=8.5   peak=80.6° at 00:28 moon_sep=65.2°
  NGC6210    nan                       type=PN    mag=9.65  peak=80.5° at 19:54 moon_sep=68.3°
  NGC7662    Copeland's Blue Snowball  type=PN    mag=8.3   peak=69.3° at 01:02 moon_s